# IMPORT

In [88]:
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [89]:
!pip install python-binance

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [90]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

import yfinance as yf
import matplotlib.pyplot as plt

import dotenv
import binance


## Import Tether from Binance

In [91]:
from binance.client import Client
import pandas as pd
from datetime import datetime


client = Client()

symbol = "BTCUSDT"
interval = Client.KLINE_INTERVAL_1DAY
start_date = "1 Jan, 2017"
end_date = datetime.today().strftime("%d %b, %Y")

klines = client.get_historical_klines(
    symbol,
    interval,
    start_date,
    end_date
)

columns = [
    "Open time",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Close time",
    "Quote Asset Volume",
    "Number of Trades",
    "Taker Buy Base Asset Volume",
    "Taker Buy Quote Asset Volume",
    "Ignore"
]

df = pd.DataFrame(klines, columns=columns)

# Convert timestamps
df["Open time"] = pd.to_datetime(df["Open time"], unit="ms")
df["Close time"] = pd.to_datetime(df["Close time"], unit="ms")

# Colonnes numériques
columns_to_convert = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'Quote Asset Volume', 'Number of Trades',
    'Taker Buy Base Asset Volume',
    'Taker Buy Quote Asset Volume'
]

df[columns_to_convert] = df[columns_to_convert].astype(float)

df = df.drop(columns=["Ignore"])
df = df.set_index("Open time")

print("Rows:", len(df))
print(df.head())

Rows: 3122
               Open     High      Low    Close       Volume  \
Open time                                                     
2017-08-17  4261.48  4485.39  4200.74  4285.08   795.150377   
2017-08-18  4285.08  4371.52  3938.77  4108.37  1199.888264   
2017-08-19  4108.37  4184.69  3850.00  4139.98   381.309763   
2017-08-20  4120.98  4211.08  4032.62  4086.29   467.083022   
2017-08-21  4069.13  4119.62  3911.79  4016.00   691.743060   

                        Close time  Quote Asset Volume  Number of Trades  \
Open time                                                                  
2017-08-17 2017-08-17 23:59:59.999        3.454770e+06            3427.0   
2017-08-18 2017-08-18 23:59:59.999        5.086958e+06            5233.0   
2017-08-19 2017-08-19 23:59:59.999        1.549484e+06            2153.0   
2017-08-20 2017-08-20 23:59:59.999        1.930364e+06            2321.0   
2017-08-21 2017-08-21 23:59:59.999        2.797232e+06            3972.0   

            Ta

# CONFIG

In [92]:
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_FRAC = 0.8

# Trading / reward parameters
FEE = 0.0005         # transaction cost per unit position change
KAPPA = 0.1          # risk penalty weight
INITIAL_BUDGET = 100000.0
MAX_LEVERAGE = 2.0

# PPO hyperparameters - OPTIMIZED
num_envs = 4  # Reduced from 16 for stability
n_steps = 128
total_updates = 1000  # Reduced from 2000

gamma = 0.99
gae_lambda = 0.95

lr = 1e-4  # Slower learning (was 3e-4)
vf_coef = 0.5
ent_coef = 0.01  # More exploration (was 0.001)
max_grad_norm = 0.5

clip_eps = 0.1  # More conservative (was 0.2)
ppo_epochs = 4  # Reduced from 10
minibatch_size = 32  # Reduced from 64
target_kl = 0.1

# HELPER FUNCTIONS AND CLASSES

In [93]:
from binance.client import Client
import pandas as pd
from datetime import datetime, timezone

def load_ohlcv(ticker="BTCUSDT", start="1 Jan, 2017", end=None, interval="1d"):

    client = Client()

    # Mapping interval
    interval_map = {
        "1d": Client.KLINE_INTERVAL_1DAY,
        "1h": Client.KLINE_INTERVAL_1HOUR,
        "4h": Client.KLINE_INTERVAL_4HOUR
    }

    binance_interval = interval_map.get(interval, Client.KLINE_INTERVAL_1DAY)

    # Si pas de end → jusqu’à aujourd’hui
    if end is None:
        end = datetime.now(timezone.utc).strftime("%d %b, %Y")

    klines = client.get_historical_klines(
        ticker,
        binance_interval,
        start,
        end
    )

    columns = [
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_asset_volume",
        "number_of_trades",
        "taker_buy_base",
        "taker_buy_quote",
        "ignore"
    ]

    df = pd.DataFrame(klines, columns=columns)

    # Date index
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df = df.set_index("open_time")
    df.index.name = "Date"

    # Garder uniquement les colonnes du screenshot
    df = df[["close", "high", "low", "open", "volume"]]

    # Convertir en float
    df = df.astype(float)

    return df


# Usage
df = load_ohlcv()
df.head()


,close,high,low,open,volume
Date,,,,,
2017-08-17,4285.08,4485.39,4200.74,4261.48,795.150377
2017-08-18,4108.37,4371.52,3938.77,4285.08,1199.888264
2017-08-19,4139.98,4184.69,3850.00,4108.37,381.309763
2017-08-20,4086.29,4211.08,4032.62,4120.98,467.083022
2017-08-21,4016.00,4119.62,3911.79,4069.13,691.743060


# FEATURE ENGINEERING WITH BITCOIN INDICATORS

In [94]:
def add_features_and_forecast(df, ewma_span=20, vol_window=20):
    df = df.copy()
    df["log_close"] = np.log(df["close"])
    df["r"] = df["log_close"].diff()

    # Original features
    df["mu_hat"] = df["r"].ewm(span=ewma_span, adjust=False).mean()
    df["sigma_hat"] = df["r"].rolling(vol_window).std()
    df["r_lag1"] = df["r"].shift(1)

    # === BITCOIN-OPTIMIZED TECHNICAL FEATURES ===

    # RSI (14-period)
    def compute_rsi(series, period=14):
        delta = series.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / (loss + 1e-8)
        return (100 - (100 / (1 + rs))) / 100.0

    df["rsi"] = compute_rsi(df["close"], period=14)

    # MACD
    ema_12 = df["close"].ewm(span=12, adjust=False).mean()
    ema_26 = df["close"].ewm(span=26, adjust=False).mean()
    df["macd"] = ema_12 - ema_26
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_norm"] = df["macd"] / (df["close"] + 1e-8)

    # Bollinger Bands
    sma_20 = df["close"].rolling(window=20).mean()
    std_20 = df["close"].rolling(window=20).std()
    df["bb_position"] = (df["close"] - (sma_20 - 2*std_20)) / (4*std_20 + 1e-8)
    df["bb_position"] = df["bb_position"].clip(0, 1)

    # SMA 20 & 200
    df["sma_20"] = sma_20
    df["sma_200"] = df["close"].rolling(window=200).mean()
    df["sma_ratio"] = (df["sma_20"] / (df["sma_200"] + 1e-8) - 1.0).clip(-0.5, 0.5)

    # ATR
    high_low = df["high"] - df["low"]
    high_close = abs(df["high"] - df["close"].shift())
    low_close = abs(df["low"] - df["close"].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["atr_ratio"] = (true_range.rolling(14).mean() / (df["close"] + 1e-8)).clip(0, 0.05)

    # Volume
    df["volume_ma_20"] = df["volume"].rolling(window=20).mean()
    df["volume_ratio"] = np.log1p((df["volume"] / (df["volume_ma_20"] + 1e-8)).clip(0, 5))

    # Golden Cross
    df["golden_cross"] = (df["sma_20"] > df["sma_200"]).astype(float)

    # Momentum
    df["momentum_5"] = df["close"].pct_change(5).clip(-0.1, 0.1)

    df = df.dropna()
    return df

df_feat = add_features_and_forecast(df)
print(f"✅ {len(df_feat.columns)} features created")
df_feat.head()

✅ 23 features created


,close,high,low,open,volume,log_close,r,mu_hat,sigma_hat,r_lag1,...,macd_norm,bb_position,sma_20,sma_200,sma_ratio,atr_ratio,volume_ma_20,volume_ratio,golden_cross,momentum_5
Date,,,,,,,,,,,,,,,,,,,,,
2018-03-04,11515.00,11565.0,11050.02,11464.47,17295.918653,9.351406,0.004397,0.010804,0.051823,0.037819,...,0.025015,0.872778,10387.4700,8756.49030,0.186260,0.05,40223.467409,0.357671,1.0,0.089503
2018-03-05,11454.00,11710.0,11415.01,11515.00,15144.231063,9.346094,-0.005312,0.009269,0.050430,0.004397,...,0.028609,0.851204,10533.1750,8792.33490,0.197995,0.05,39207.930314,0.326605,1.0,0.100000
2018-03-06,10716.48,11455.0,10555.48,11455.00,29515.572363,9.279538,-0.066556,0.002048,0.049212,-0.005312,...,0.027637,0.549618,10596.4995,8825.37545,0.200685,0.05,38643.111289,0.567470,1.0,-0.018637
2018-03-07,9910.00,10899.0,9389.31,10716.48,50647.671080,9.201300,-0.078238,-0.005599,0.051156,-0.066556,...,0.020563,0.220271,10591.9950,8854.22555,0.196264,0.05,38554.115007,0.838838,1.0,-0.100000
2018-03-08,9271.64,10099.0,9060.00,9910.00,41109.473226,9.134716,-0.066584,-0.011407,0.053059,-0.078238,...,0.008428,0.025205,10547.5780,8880.15230,0.187770,0.05,38701.528370,0.723782,1.0,-0.100000


## Train/Test split (time-based)

In [95]:
n = len(df_feat)
split = int(TRAIN_FRAC * n)

df_train = df_feat.iloc[:split].reset_index(drop=True)
df_test  = df_feat.iloc[split:].reset_index(drop=True)

print(len(df_train), len(df_test))

2338 585


## Trading Environment (target position action)

In [96]:
class TradingEnv(gym.Env):
    """
    Enhanced trading environment with:
    - Leverage & PnL tracking
    - Budget & Liquidity management
    - Long/Short mechanism with refined rewards
    - Bitcoin-optimized state
    """
    metadata = {"render_modes": []}

    def __init__(self, df, fee=0.0005, kappa=0.1, initial_budget=100000.0, max_leverage=2.0):
        super().__init__()
        self.df = df.reset_index(drop=True)  # Critical: reset index!
        self.fee = float(fee)
        self.kappa = float(kappa)
        self.initial_budget = float(initial_budget)
        self.max_leverage = float(max_leverage)

        # Action = target position in [-1, 1]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)

        # Market + Technical features
        self.feature_cols = [
            "r", "r_lag1", "mu_hat", "sigma_hat",
            "rsi", "macd_norm", "bb_position",
            "sma_ratio", "atr_ratio", "volume_ratio",
            "golden_cross", "momentum_5"
        ]
        # Only use features that exist
        self.feature_cols = [f for f in self.feature_cols if f in df.columns]

        # obs_dim = market_features + portfolio_features(11)
        obs_dim = len(self.feature_cols) + 11
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = 1
        self.pos = 0.0  # Position: -1 (short) to +1 (long)
        self.equity = self.initial_budget
        self.peak = self.initial_budget
        self.cumulative_profit = 0.0
        self.cost_accumulated = 0.0
        self.cumulative_returns = 0.0
        self.trades_made = 0
        self.winning_trades = 0
        self.prev_pos = 0.0  # Track previous position for logging
        self.trade_log = []  # Log all trades
        return self._get_obs(), {}

    def _get_obs(self):
        """Enhanced observation with 25 features"""

        # Clamp time index
        t_idx = int(np.clip(self.t, 0, len(self.df) - 1))

        # Market features
        market_features = []
        for col in self.feature_cols:
            try:
                val = float(self.df.iloc[t_idx][col])
                val = 0.0 if not np.isfinite(val) else val
            except:
                val = 0.0
            market_features.append(val)

        x = np.array(market_features, dtype=np.float32)

        # Portfolio features (11 total)
        equity_norm = float(self.equity / self.initial_budget)
        drawdown = float((self.peak - self.equity) / max(self.peak, 1.0))

        # Budget & Liquidity
        position_value = abs(self.pos) * self.equity
        free_capital = max(0.0, self.equity - position_value)
        liquidity_ratio = float(free_capital / max(self.equity, 1.0))

        # Leverage
        leverage = float(abs(self.pos) * self.max_leverage)

        # PnL
        try:
            r_t = float(self.df.iloc[t_idx]["r"])
            r_t = 0.0 if not np.isfinite(r_t) else r_t
        except:
            r_t = 0.0
        unrealized_pnl = float(self.pos * r_t)

        # Cumulative metrics
        cumulative_returns = float(self.cumulative_returns)
        win_rate = float(self.winning_trades / max(self.trades_made, 1.0)) if self.trades_made > 0 else 0.0
        costs_norm = float(min(1.0, self.cost_accumulated / max(self.initial_budget, 1.0)))
        momentum_signal = float(0.0)

        portfolio_features = np.array([
            self.pos, equity_norm, drawdown, liquidity_ratio, leverage,
            unrealized_pnl, cumulative_returns, win_rate,
            costs_norm, momentum_signal, 0.0  # padding
        ], dtype=np.float32)

        obs = np.concatenate([x, portfolio_features])
        return obs

    def step(self, action):
        # Safety check
        if self.t >= len(self.df) - 1:
            return self._get_obs(), 0.0, True, False, {
                "cumulative_profit": self.cumulative_profit,
                "equity": self.equity,
                "position": self.pos,
                "costs": self.cost_accumulated
            }

        # === REWARD STRUCTURE (COMPLETELY FIXED) ===
        # CRITICAL: Reward is PURE LEARNING SIGNAL, NOT connected to equity_update!
        # Reward should reflect what happened in this step for PPO learning
        # Equity update should be based on ACTUAL dollar P&L

        # Action
        a = float(np.clip(action[0], -1.0, 1.0))

        # Get market data
        r_t = float(self.df.iloc[self.t]["r"])
        sigma_t = float(self.df.iloc[self.t]["sigma_hat"])
        if not np.isfinite(sigma_t):
            sigma_t = 0.01

        # 1. Core PnL signal from position
        pnl_return = self.pos * r_t  # ±0.01 to ±0.02

        # 2. Scale for PPO learning (rewards ~±1 ideal)
        pnl_reward_scaled = 100.0 * pnl_return

        # 3. Transaction cost (for learning signal only - tiny)
        position_change = abs(a - self.pos)
        cost_penalty_signal = 0.01 * self.fee * position_change

        # 4. Risk penalty (for learning signal - even tinier)
        risk_penalty_signal = 0.001 * (a ** 2) * sigma_t

        # Total reward for PPO (NOT related to equity update!)
        reward = pnl_reward_scaled - cost_penalty_signal - risk_penalty_signal

        # === UPDATE EQUITY (SEPARATE from reward!) ===
        # This is the ACTUAL dollar P&L
        pnl_dollars = self.pos * self.equity * r_t

        # ACTUAL transaction cost in dollars (small, realistic)
        actual_cost_dollars = self.fee * position_change * self.equity

        # Update equity with ACTUAL P&L (not scaled reward!)
        self.equity = self.equity + pnl_dollars - actual_cost_dollars

        # Safety: prevent negative equity
        if self.equity < 0:
            self.equity = self.initial_budget * 0.001
            reward = -1.0

        # === TRACKING (use actual dollar values!) ===
        self.cumulative_profit += pnl_dollars - actual_cost_dollars
        self.cost_accumulated += actual_cost_dollars
        self.cumulative_returns += pnl_return  # Track actual return

        if position_change > 0.01:
            self.trades_made += 1
            if pnl_dollars > 0:
                self.winning_trades += 1
            self.position_entry_time = self.t

            # === COLLECT TRADE LOG (No live printing) ===
            position_type = "LONG" if a > 0.1 else "SHORT" if a < -0.1 else "FLAT"
            prev_type = "LONG" if self.prev_pos > 0.1 else "SHORT" if self.prev_pos < -0.1 else "FLAT"


            # Store in trade log
            self.trade_log.append({
                "day": self.t,
                "prev_type": prev_type,
                "new_type": position_type,
                "prev_pos": float(self.prev_pos),
                "new_pos": float(a),
                "daily_return": float(r_t),
                "pnl_dollars": float(pnl_dollars),
                "equity": float(self.equity),
                "reward": float(reward)
            })

        # Update position tracking
        self.prev_pos = a

        # === UPDATE STATE ===
        self.pos = a
        self.peak = max(self.peak, self.equity)

        # === TIME & TERMINATION ===
        self.t += 1
        terminated = (self.t >= len(self.df) - 1) or (self.equity <= 0)
        truncated = False

        next_obs = self._get_obs()

        return next_obs, float(reward), terminated, truncated, {
            "cumulative_profit": self.cumulative_profit,
            "equity": self.equity,
            "position": self.pos,
            "costs": self.cost_accumulated
        }

## Vectorized env (train)

In [97]:
# Add constants
INITIAL_BUDGET = 100000.0
MAX_LEVERAGE = 2.0

def make_env(df):
    def thunk():
        return TradingEnv(df, fee=FEE, kappa=KAPPA, initial_budget=INITIAL_BUDGET, max_leverage=MAX_LEVERAGE)
    return thunk

env = gym.vector.SyncVectorEnv([make_env(df_train) for _ in range(num_envs)])
obs_dim = env.single_observation_space.shape[0]
act_dim = env.single_action_space.shape[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("obs_dim:", obs_dim, "act_dim:", act_dim, "device:", device)

obs_dim: 23 act_dim: 1 device: cpu


## PPO model (Gaussian policy + tanh squash + corrected logprob)

In [98]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh()
        )
        self.mu = nn.Linear(128, act_dim)
        self.log_std = nn.Parameter(torch.ones(act_dim) * -1.0)  # good start
        self.v = nn.Linear(128, 1)

    def forward(self, obs):
        x = self.net(obs)
        mu = self.mu(x)
        std = torch.exp(self.log_std)
        dist = Normal(mu, std)
        value = self.v(x).squeeze(-1)
        return dist, value

def squash(u):
    return torch.tanh(u)  # maps to [-1,1]
# main formula:
# a = f(u)
# log p(a) = log p(u) - log |det(Jacobian)|
# log p(a) ist die gesuchte policy log pi(a|a)

# we have f = tanh
# a = tanh(u)
# da/du = 1 - tanh(u)^2
# da/du = 1 - a²
# we need: log |det(Jacobian)|
# we get: log |det(Jacobian)| = log(1 - tanh(u)^2)
# in code: log_det = torch.log(1.0 - torch.tanh(u).pow(2) + eps).sum(-1)

def logprob_squashed(dist, u):
    # log p(u)
    logp_u = dist.log_prob(u).sum(-1)
    # change-of-variables for tanh
    eps = 1e-6
    log_det = torch.log(1.0 - torch.tanh(u).pow(2) + eps).sum(-1)
    return logp_u - log_det

## GAE

In [99]:
def compute_gae(rewards, dones, values, last_value, gamma=0.99, lam=0.95):
    """
    rewards: [T, N]
    dones:   [T, N] (1.0 means terminal boundary for bootstrap mask)
    values:  [T, N]
    last_value: [N]
    """
    T, N = rewards.shape
    adv = torch.zeros(T, N, device=values.device)
    gae = torch.zeros(N, device=values.device)

    for t in reversed(range(T)):
        not_done = 1.0 - dones[t]
        next_value = last_value if t == T - 1 else values[t + 1]
        delta = rewards[t] + gamma * next_value * not_done - values[t]
        gae = delta + gamma * lam * not_done * gae
        adv[t] = gae

    returns = adv + values
    return returns, adv

# TRAINING

## PPO training loop

In [ ]:
model = ActorCritic(obs_dim, act_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

obs, _ = env.reset(seed=SEED)
obs = torch.as_tensor(obs, dtype=torch.float32, device=device)

ep_returns = np.zeros(num_envs, dtype=np.float32)
ep_history = []

for update in range(total_updates):
    # Rollout buffers
    obs_buf  = torch.zeros(n_steps, num_envs, obs_dim, device=device)
    u_buf    = torch.zeros(n_steps, num_envs, act_dim, device=device)
    logp_buf = torch.zeros(n_steps, num_envs, device=device)
    rew_buf  = torch.zeros(n_steps, num_envs, device=device)
    done_buf = torch.zeros(n_steps, num_envs, device=device)
    val_buf  = torch.zeros(n_steps, num_envs, device=device)

    for t in range(n_steps):
        obs_buf[t] = obs
        with torch.no_grad():
          dist, value = model(obs)
          u = dist.sample()
          a = squash(u)
          logp = logprob_squashed(dist, u)

        u_buf[t] = u
        logp_buf[t] = logp.detach()
        val_buf[t] = value.detach()

        next_obs, reward, terminated, truncated, infos = env.step(a.detach().cpu().numpy())
        done_env = np.logical_or(terminated, truncated)
        done_boot = terminated  # bootstrap mask (for time-limit envs you may choose terminated only)

        rew_buf[t] = torch.as_tensor(reward, dtype=torch.float32, device=device)
        done_buf[t] = torch.as_tensor(done_boot, dtype=torch.float32, device=device)

        # Episode return tracking
        ep_returns += reward
        if done_env.any():
            finished = np.where(done_env)[0]
            ep_history.extend(ep_returns[finished].tolist())
            ep_returns[finished] = 0.0

        obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)

    # Bootstrap last value
    with torch.no_grad():
        _, last_value = model(obs)

    returns, adv = compute_gae(rew_buf, done_buf, val_buf, last_value, gamma=gamma, lam=gae_lambda)

    # Flatten
    B = n_steps * num_envs
    obs_batch  = obs_buf.reshape(B, obs_dim)
    u_batch    = u_buf.reshape(B, act_dim)
    old_logp   = logp_buf.reshape(B)
    old_value  = val_buf.reshape(B)      # important for value clipping if you add it later
    ret_batch  = returns.reshape(B).detach()
    adv_batch  = adv.reshape(B).detach()

    # Advantage normalization
    adv_batch = (adv_batch - adv_batch.mean()) / (adv_batch.std() + 1e-8)

    idx = torch.arange(B, device=device)
    stop = False

    for _ in range(ppo_epochs):
        perm = idx[torch.randperm(B)]
        for start in range(0, B, minibatch_size):
            mb = perm[start:start + minibatch_size]

            dist, value = model(obs_batch[mb])
            logp = logprob_squashed(dist, u_batch[mb])
            entropy = dist.entropy().sum(-1)

            # Early stop by approximate KL (minibatch estimate)
            approx_kl = (old_logp[mb] - logp).mean().detach()
            if approx_kl.item() > target_kl:
                stop = True
                break

            ratio = torch.exp(logp - old_logp[mb])

            # Clipped policy objective
            unclipped = ratio * adv_batch[mb]
            clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv_batch[mb]
            policy_loss = -torch.min(unclipped, clipped).mean()

            # Value loss (simple version; you can add value clipping later)
            value_loss = (ret_batch[mb] - value).pow(2).mean()

            # Entropy bonus
            entropy_loss = -entropy.mean()

            loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        if stop:
            break

        # Keep std in a sane range (helps prevent wild exploration collapse/explosion)
        with torch.no_grad():
            model.log_std.clamp_(-2.0, -0.5)

    if update % 100 == 0:
        mean_100 = np.mean(ep_history[-100:]) if len(ep_history) >= 100 else np.nan
        print(f"Update {update:4d} | mean_return(last100) {mean_100:8.1f} | log_std {model.log_std.data.cpu().numpy()}")

Update    0 | mean_return(last100)      nan | log_std [-1.0021694]
Update  100 | mean_return(last100)      nan | log_std [-0.9856729]
Update  200 | mean_return(last100)      nan | log_std [-0.96593904]
Update  300 | mean_return(last100)      nan | log_std [-0.95970803]
Update  400 | mean_return(last100)      nan | log_std [-0.9542431]


# EVALUATION

## Evaluation on test set (single env, deterministic actions)

In [ ]:
def eval_policy(model, df_eval, episodes=5):
    env_eval = TradingEnv(df_eval, fee=FEE, kappa=KAPPA, initial_budget=INITIAL_BUDGET, max_leverage=MAX_LEVERAGE)
    returns = []

    for _ in range(episodes):
        obs, _ = env_eval.reset()
        done = False
        ep_ret = 0.0

        while not done:
            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                dist, _ = model(obs_t)
                # Deterministic: use mean action (mu), then squash
                u = dist.mean
                a = squash(u).cpu().numpy()[0]

            obs, reward, terminated, truncated, _ = env_eval.step(a)
            done = terminated or truncated
            ep_ret += reward

        returns.append(ep_ret)

    return float(np.mean(returns))

test_score = eval_policy(model, df_test, episodes=10)
print("EVAL mean episode reward:", test_score)

# PLOT

## Equity curve plot (test set, one run)

In [ ]:
def run_equity_curve(model, df_eval):
    env_eval = TradingEnv(df_eval, fee=FEE, kappa=KAPPA, initial_budget=INITIAL_BUDGET, max_leverage=MAX_LEVERAGE)
    obs, _ = env_eval.reset()
    done = False

    equity = [env_eval.equity]
    pos_hist = [env_eval.pos]
    profit_hist = [0.0]
    cost_hist = [0.0]

    while not done:
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            dist, _ = model(obs_t)
            u = dist.mean
            a = squash(u).cpu().numpy()[0]

        obs, reward, terminated, truncated, info = env_eval.step(a)
        done = terminated or truncated
        equity.append(env_eval.equity)
        pos_hist.append(env_eval.pos)
        profit_hist.append(info["cumulative_profit"])
        cost_hist.append(info["costs"])

    # Return results + trade log for analysis
    return np.array(equity), np.array(pos_hist), np.array(profit_hist), np.array(cost_hist), env_eval.trade_log

equity, pos_hist, profit_hist, cost_hist, trade_log = run_equity_curve(model, df_test)

# Create 4-panel plot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Equity Curve
axes[0, 0].plot(equity, linewidth=2.5, color='green', label='Equity')
axes[0, 0].fill_between(range(len(equity)), INITIAL_BUDGET, equity, alpha=0.3, color='green')
axes[0, 0].axhline(y=INITIAL_BUDGET, color='red', linestyle='--', linewidth=2, label='Initial Budget')
axes[0, 0].set_title("Equity Curve (Test Set)", fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel("Trading Days", fontsize=12)
axes[0, 0].set_ylabel("Equity (USDT)", fontsize=12)
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Panel 2: Cumulative Profit
axes[0, 1].plot(profit_hist, linewidth=2.5, color='blue', label='Profit')
axes[0, 1].fill_between(range(len(profit_hist)), 0, profit_hist, alpha=0.3, color='blue')
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 1].set_title("Cumulative Profit (Test Set)", fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel("Trading Days", fontsize=12)
axes[0, 1].set_ylabel("Profit (USDT)", fontsize=12)
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Panel 3: Position History
colors = ['red' if p < 0 else 'green' for p in pos_hist]
axes[1, 0].bar(range(len(pos_hist)), pos_hist, color=colors, alpha=0.6, width=1.0)
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[1, 0].set_title("Position History (Test Set)", fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel("Trading Days", fontsize=12)
axes[1, 0].set_ylabel("Position [-1 Short, +1 Long]", fontsize=12)
axes[1, 0].set_ylim([-1.2, 1.2])
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Panel 4: Cumulative Costs
axes[1, 1].plot(cost_hist, linewidth=2.5, color='red', label='Transaction Costs')
axes[1, 1].fill_between(range(len(cost_hist)), 0, cost_hist, alpha=0.3, color='red')
axes[1, 1].set_title("Cumulative Transaction Costs (Test Set)", fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel("Trading Days", fontsize=12)
axes[1, 1].set_ylabel("Costs (USDT)", fontsize=12)
axes[1, 1].legend(fontsize=11)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

# Print detailed statistics
print("=" * 70)
print("TRADING PERFORMANCE SUMMARY")
print("=" * 70)
final_equity = equity[-1]
total_profit = profit_hist[-1]
total_costs = cost_hist[-1]
return_pct = (final_equity - INITIAL_BUDGET) / INITIAL_BUDGET * 100
max_drawdown = (1 - (equity.min() / INITIAL_BUDGET)) * 100
sharpe = np.std(np.diff(equity)) / (np.mean(np.diff(equity)) + 1e-8) if np.mean(np.diff(equity)) != 0 else 0
win_rate = np.sum(np.diff(equity) > 0) / len(np.diff(equity)) * 100 if len(np.diff(equity)) > 0 else 0

print(f"Initial Budget:         ${INITIAL_BUDGET:>15,.2f}")
print(f"Final Equity:           ${final_equity:>15,.2f}")
print(f"Total Profit:           ${total_profit:>15,.2f}")
print(f"Total Costs:            ${total_costs:>15,.2f}")
print(f"Return:                 {return_pct:>15.2f}%")
print(f"Max Drawdown:           {max_drawdown:>15.2f}%")
print(f"Sharpe Ratio:           {sharpe:>15.4f}")
print(f"Win Rate:               {win_rate:>15.2f}%")
print(f"Trading Days:           {len(equity):>15}")
print("=" * 70)

# === TRADE LOG SUMMARY ===
print("\n" + "=" * 120)
print("DETAILED TRADE LOG - ALL LONG/SHORT TRANSITIONS")
print("=" * 120)

if len(trade_log) > 0:
    print(f"{'Day':>5} | {'From':>6} | {'To':>6} | {'Pos':>8} | {'Return':>8} | {'PnL':>12} | {'Equity':>15} | {'Reward':>8}")
    print("-" * 120)

    long_count = 0
    short_count = 0
    flat_count = 0
    winning_trades = 0

    for trade in trade_log:
        print(f"{trade['day']:5d} | {trade['prev_type']:>6s} | {trade['new_type']:>6s} | "
              f"{trade['new_pos']:+7.3f} | {trade['daily_return']:+7.4f} | "
              f"${trade['pnl_dollars']:>10,.0f} | ${trade['equity']:>14,.0f} | {trade['reward']:+7.3f}")

        # Count positions
        if trade['new_type'] == 'LONG':
            long_count += 1
        elif trade['new_type'] == 'SHORT':
            short_count += 1
        elif trade['new_type'] == 'FLAT':
            flat_count += 1

        if trade['pnl_dollars'] > 0:
            winning_trades += 1

    print("-" * 120)
    print(f"\n📊 TRADE SUMMARY:")
    print(f"   Total Trades:        {len(trade_log)}")
    print(f"   → LONG positions:    {long_count}")
    print(f"   → SHORT positions:   {short_count}")
    print(f"   → FLAT positions:    {flat_count}")
    print(f"   Winning Trades:      {winning_trades} ({winning_trades/len(trade_log)*100:.1f}%)")
    print(f"   Losing Trades:       {len(trade_log)-winning_trades} ({(len(trade_log)-winning_trades)/len(trade_log)*100:.1f}%)")
    print("=" * 120)
else:
    print("No trades recorded.")
    print("=" * 120)